# Prática 01 — Caminho Mínimo (Versão Modular)

Este notebook segue a mesma estrutura do `pratica01_final.ipynb`, mas implementa as letras (a)–(e) como **funções reutilizáveis** para serem chamadas na letra (f).

- Parte 1: Dijkstra — funções (a) a (e)
- Parte 2: Heurística Gulosa — funções (a) a (e) (a/b reutilizam geração e salvamento)
- Letra (f): aplica as funções em instâncias grandes (10k e 1M) e gera comparações

---

In [2]:
# Imports e parâmetros
from typing import Dict, List, Tuple, Iterable
from collections import deque
from math import inf
import heapq, random, os
import matplotlib.pyplot as plt
import numpy as np

# Parâmetros padrão
N_MIN = 4
N_MAX = 10
SEED = 42
ORIGEM = 0

# Pastas de saída (mantém padronização do projeto)
PASTA_D = 'resultados/dijkstra'
PASTA_G = 'resultados/gulosa'
PASTA_C = 'resultados/comparacao'
for p in [
    PASTA_D,
    f"{PASTA_D}/img",
    f"{PASTA_D}/instancias",
    PASTA_G,
    f"{PASTA_G}/img",
    f"{PASTA_G}/instancias",
    PASTA_C,
]:
    os.makedirs(p, exist_ok=True)

print('✓ Ambiente inicializado')

✓ Ambiente inicializado


---
# Funções Comuns (utilitárias)
---

In [3]:
def gerar_grafo_completo(n: int, peso_min: float = 1.0, peso_max: float = 10.0, seed: int | None = None) -> Dict[int, List[Tuple[int, float]]]:
    if seed is not None:
        random.seed(seed)
    adj = {u: [] for u in range(n)}
    for u in range(n):
        for v in range(n):
            if u != v:
                peso = random.uniform(peso_min, peso_max)
                adj[u].append((v, peso))
    return adj

def salvar_grafo_txt(adj: Dict[int, List[Tuple[int, float]]], caminho: str) -> None:
    os.makedirs(os.path.dirname(caminho) or '.', exist_ok=True)
    with open(caminho, 'w', encoding='utf-8') as f:
        n = len(adj)
        f.write(f'Grafo completo (dirigido) com n={n} vertices\n')
        f.write('Formato: u -> [(v, peso), ...]\n')
        f.write('='*60 + '\n')
        for u in sorted(adj):
            arestas = ', '.join([f'({v}, {peso:.4f})' for v, peso in adj[u]])
            f.write(f'{u} -> [{arestas}]\n')

def carregar_grafo_ewd(caminho: str) -> Dict[int, List[Tuple[int, float]]]:
    with open(caminho, 'r', encoding='utf-8') as f:
        V = int(f.readline().strip())
        E = int(f.readline().strip())
        adj = {i: [] for i in range(V)}
        for _ in range(E):
            linha = f.readline().strip()
            if not linha: break
            u, v, peso = linha.split()[:3]
            adj[int(u)].append((int(v), float(peso)))
    return adj

print('✓ Funções utilitárias definidas')

✓ Funções utilitárias definidas


---
# Parte 1 — Dijkstra: (a)–(e) como funções
---

In [5]:
def dijkstra(adj: Dict[int, List[Tuple[int, float]]], origem: int = 0):
    if not adj or origem not in adj:
        raise ValueError('Grafo vazio ou origem inválida')
    dist = {u: inf for u in adj}
    parent = {u: None for u in adj}
    dist[origem] = 0.0
    pq = [(0.0, origem)]
    comparacoes = 0
    while pq:
        dist_u, u = heapq.heappop(pq)
        if dist_u != dist[u]:
            continue
        for v, peso in adj[u]:
            comparacoes += 1
            nova = dist[u] + peso
            if nova < dist[v]:
                dist[v] = nova
                parent[v] = u
                heapq.heappush(pq, (nova, v))
    return dist, parent, comparacoes

# (a) Gerar grafos completos

def p1_a_gerar_grafos(ns: Iterable[int], seed: int | None = None) -> List[Dict[int, List[Tuple[int, float]]]]:
    return [gerar_grafo_completo(n, seed=seed) for n in ns]

# (b) Salvar grafos

def p1_b_salvar_grafos(ns: Iterable[int], grafos: List[Dict[int, List[Tuple[int, float]]]], pasta: str = PASTA_D) -> None:
    os.makedirs(pasta, exist_ok=True)
    for n, g in zip(ns, grafos):
        salvar_grafo_txt(g, os.path.join(pasta, f'grafo_completo_n{n}.txt'))

# (c) Aplicar Dijkstra

def p1_c_aplicar_dijkstra(grafos: List[Dict[int, List[Tuple[int, float]]]], origem: int = 0):
    return [dijkstra(g, origem=origem) for g in grafos]

# (d) Salvar comparações

def p1_d_salvar_comparacoes(ns: List[int], resultados: List[Tuple[dict, dict, int]], caminho: str = f'{PASTA_D}/teste_comparacoes.txt') -> List[int]:
    comps = [r[2] for r in resultados]
    os.makedirs(os.path.dirname(caminho) or '.', exist_ok=True)
    with open(caminho, 'w', encoding='utf-8') as f:
        f.write('n\tcomparacoes\n')
        for n, c in zip(ns, comps):
            f.write(f'{n}\t{c}\n')
    return comps

# (e) Plotar gráfico n vs comparações

def p1_e_plotar(ns: List[int], comps: List[int], caminho_img: str = f'{PASTA_D}/img/teste_comparacoes.png') -> None:
    os.makedirs(os.path.dirname(caminho_img) or '.', exist_ok=True)
    plt.figure(figsize=(8, 5))
    plt.plot(ns, comps, marker='o', linewidth=2, markersize=6)
    plt.xlabel('Número de vértices (n)')
    plt.ylabel('Número de comparações')
    plt.title('Dijkstra - Teste em Grafos Completos')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(caminho_img, dpi=150)
    plt.close()

print('✓ Parte 1 (Dijkstra) — funções definidas (a–e)')

✓ Parte 1 (Dijkstra) — funções definidas (a–e)


---
# Parte 2 — Heurística Gulosa: (a)–(e) como funções
---

In [7]:
def heuristica_gulosa(adj: Dict[int, List[Tuple[int, float]]], origem: int = 0):
    if origem not in adj:
        raise ValueError('Origem inválida')
    dist = {u: inf for u in adj}
    parent = {u: None for u in adj}
    dist[origem] = 0.0
    visited = {origem}
    fila = deque([origem])
    comparacoes = 0
    while fila:
        u = fila.popleft()
        melhor_v, melhor_custo = None, inf
        for v, peso in adj[u]:
            comparacoes += 1
            if v in visited:
                continue
            custo = dist[u] + peso
            if custo < dist[v] and custo < melhor_custo:
                melhor_custo, melhor_v = custo, v
        if melhor_v is not None:
            dist[melhor_v] = melhor_custo
            parent[melhor_v] = u
            visited.add(melhor_v)
            fila.append(melhor_v)
    return dist, parent, comparacoes

# (a) e (b) reutilizam as funções comuns: p1_a_gerar_grafos / p1_b_salvar_grafos

# (c) Aplicar Gulosa

def p2_c_aplicar_gulosa(grafos: List[Dict[int, List[Tuple[int, float]]]], origem: int = 0):
    return [heuristica_gulosa(g, origem=origem) for g in grafos]

# (d) Salvar comparações

def p2_d_salvar_comparacoes(ns: List[int], resultados: List[Tuple[dict, dict, int]], caminho: str = f'{PASTA_G}/teste_comparacoes.txt') -> List[int]:
    comps = [r[2] for r in resultados]
    os.makedirs(os.path.dirname(caminho) or '.', exist_ok=True)
    with open(caminho, 'w', encoding='utf-8') as f:
        f.write('n\tcomparacoes\n')
        for n, c in zip(ns, comps):
            f.write(f'{n}\t{c}\n')
    return comps

# (e) Plotar gráfico n vs comparações

def p2_e_plotar(ns: List[int], comps: List[int], caminho_img: str = f'{PASTA_G}/img/teste_comparacoes.png') -> None:
    os.makedirs(os.path.dirname(caminho_img) or '.', exist_ok=True)
    plt.figure(figsize=(8, 5))
    plt.plot(ns, comps, marker='o', color='orange', linewidth=2, markersize=6)
    plt.xlabel('Número de vértices (n)')
    plt.ylabel('Número de comparações')
    plt.title('Heurística Gulosa - Teste em Grafos Completos')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(caminho_img, dpi=150)
    plt.close()

print('✓ Parte 2 (Gulosa) — funções definidas (a–e)')

✓ Parte 2 (Gulosa) — funções definidas (a–e)


---
# (f) Aplicação usando as funções (instâncias grandes e comparação)
---

In [8]:
def aplicar_em_instancia_grande(caminho: str, origem: int = 0, prefixo_saida: str = '10k'):
    if not os.path.exists(caminho):
        print(f'⚠️ Arquivo não encontrado: {caminho}')
        return None
    adj = carregar_grafo_ewd(caminho)
    dist_d, parent_d, comp_d = dijkstra(adj, origem=origem)
    dist_g, parent_g, comp_g = heuristica_gulosa(adj, origem=origem)
    # salvar
    os.makedirs(f'{PASTA_D}/instancias', exist_ok=True)
    os.makedirs(f'{PASTA_G}/instancias', exist_ok=True)
    with open(f'{PASTA_D}/instancias/{prefixo_saida}_dijkstra.txt', 'w', encoding='utf-8') as f:
        f.write(f'DIJKSTRA - Instância {prefixo_saida}\n')
        f.write(f'Origem: {origem}\n')
        f.write(f'Vertices: {len(adj):,}\n')
        f.write(f'Comparacoes: {comp_d:,}\n')
        f.write(f'Alcancados: {sum(1 for d in dist_d.values() if d != inf):,}\n')
    with open(f'{PASTA_G}/instancias/{prefixo_saida}_gulosa.txt', 'w', encoding='utf-8') as f:
        f.write(f'GULOSA - Instância {prefixo_saida}\n')
        f.write(f'Origem: {origem}\n')
        f.write(f'Vertices: {len(adj):,}\n')
        f.write(f'Comparacoes: {comp_g:,}\n')
        f.write(f'Alcancados: {sum(1 for d in dist_g.values() if d != inf):,}\n')
    return comp_d, comp_g

def comparar_instancias(instancias: List[str], comps_d: List[int], comps_g: List[int], caminho_img: str = f'{PASTA_C}/instancias_grandes.png'):
    os.makedirs(os.path.dirname(caminho_img) or '.', exist_ok=True)
    x = np.arange(len(instancias))
    width = 0.35
    fig, ax = plt.subplots(figsize=(10, 6))
    bars1 = ax.bar(x - width/2, comps_d, width, label='Dijkstra', color='steelblue')
    bars2 = ax.bar(x + width/2, comps_g, width, label='Gulosa', color='orange')
    ax.set_xlabel('Instância')
    ax.set_ylabel('Número de Comparações')
    ax.set_title('Comparação: Instâncias Grandes')
    ax.set_xticks(x)
    ax.set_xticklabels(instancias)
    ax.legend()
    ax.grid(True, axis='y', linestyle='--', alpha=0.6)
    for bars in [bars1, bars2]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., h, f'{int(h):,}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.savefig(caminho_img, dpi=150)
    plt.close()

print('✓ Funções (f) definidas: aplicar_em_instancia_grande, comparar_instancias')

✓ Funções (f) definidas: aplicar_em_instancia_grande, comparar_instancias


## Exemplo de uso (opcional) — Pequenos grafos
Você pode executar este bloco para gerar todo o material de teste com as funções (a)–(e).

In [ ]:
# ns = list(range(N_MIN, N_MAX + 1))
# grafos = p1_a_gerar_grafos(ns, seed=SEED)
# p1_b_salvar_grafos(ns, grafos, pasta=PASTA_D)
# res_d = p1_c_aplicar_dijkstra(grafos, origem=ORIGEM)
# comps_d = p1_d_salvar_comparacoes(ns, res_d, caminho=f'{PASTA_D}/teste_comparacoes.txt')
# p1_e_plotar(ns, comps_d, caminho_img=f'{PASTA_D}/img/teste_comparacoes.png')
#
# res_g = p2_c_aplicar_gulosa(grafos, origem=ORIGEM)
# comps_g = p2_d_salvar_comparacoes(ns, res_g, caminho=f'{PASTA_G}/teste_comparacoes.txt')
# p2_e_plotar(ns, comps_g, caminho_img=f'{PASTA_G}/img/teste_comparacoes.png')
# print('✓ Material de teste gerado com sucesso (comentado por padrão)')

## Exemplo de uso (f) — Instâncias grandes
Descomente e ajuste os caminhos abaixo para executar.

In [ ]:
# caminho_10k = r'D:\OneDrive\Pessoais\Doutorado\Cefet\022025\Teoria de Grafos\10000.txt'
# caminho_1m = r'D:\OneDrive\Pessoais\Doutorado\Cefet\022025\Teoria de Grafos\largeEWD - contains one million vertices and 15172126 edges.txt'
#
# comps_d, comps_g = [], []
# insts = []
# r10k = aplicar_em_instancia_grande(caminho_10k, origem=ORIGEM, prefixo_saida='10k')
# if r10k: insts.append('10k'); comps_d.append(r10k[0]); comps_g.append(r10k[1])
# r1m = aplicar_em_instancia_grande(caminho_1m, origem=ORIGEM, prefixo_saida='1m')
# if r1m: insts.append('1M'); comps_d.append(r1m[0]); comps_g.append(r1m[1])
# if insts:
#     comparar_instancias(insts, comps_d, comps_g, caminho_img=f'{PASTA_C}/instancias_grandes.png')
#     print('✓ Comparação de instâncias grandes gerada')
# else:
#     print('⚠️ Nenhuma instância executada (verifique caminhos)')

In [9]:
# TESTE RÁPIDO (n=4..6) — Gera, salva, aplica e plota
ns = list(range(4, 7))

# Gerar e salvar grafos (Dijkstra)
grafos = p1_a_gerar_grafos(ns, seed=SEED)
p1_b_salvar_grafos(ns, grafos, pasta=PASTA_D)

# Dijkstra
res_d = p1_c_aplicar_dijkstra(grafos, origem=ORIGEM)
comps_d = p1_d_salvar_comparacoes(ns, res_d, caminho=f'{PASTA_D}/teste_comparacoes.txt')
p1_e_plotar(ns, comps_d, caminho_img=f'{PASTA_D}/img/teste_comparacoes.png')

# Gulosa
res_g = p2_c_aplicar_gulosa(grafos, origem=ORIGEM)
comps_g = p2_d_salvar_comparacoes(ns, res_g, caminho=f'{PASTA_G}/teste_comparacoes.txt')
p2_e_plotar(ns, comps_g, caminho_img=f'{PASTA_G}/img/teste_comparacoes.png')

print('✓ Teste concluído:')
print('  - Arquivo comparações Dijkstra:', os.path.exists(f'{PASTA_D}/teste_comparacoes.txt'))
print('  - Arquivo comparações Gulosa:  ', os.path.exists(f'{PASTA_G}/teste_comparacoes.txt'))
print('  - Gráfico Dijkstra salvo?      ', os.path.exists(f'{PASTA_D}/img/teste_comparacoes.png'))
print('  - Gráfico Gulosa salvo?        ', os.path.exists(f'{PASTA_G}/img/teste_comparacoes.png'))

✓ Teste concluído:
  - Arquivo comparações Dijkstra: True
  - Arquivo comparações Gulosa:   True
  - Gráfico Dijkstra salvo?       True
  - Gráfico Gulosa salvo?         True
